<a href="https://colab.research.google.com/github/dynamodenis/QDrant/blob/main/setup/products_vectors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qdrant-client


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 4.0 MB/s eta 0:00:00


In [2]:
from qdrant_client import QdrantClient, models

In [3]:
from google.colab import userdata
client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))


In [27]:
collection_name="product_collection"

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=4, # Dimensionality of the vectors
                                       distance=models.Distance.COSINE # Distance metric for similarity search
                                       )
)
# Create payload index right after creating the collection and before uploading any data to enable filtering.
# If you add it later, HNSW won't rebuild automatically—bump ef_construct (e.g., 100→101) to trigger a safe rebuild
client.create_payload_index(
    collection_name=collection_name,
    field_name="category",
    field_schema=models.PayloadSchemaType.KEYWORD
)


UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
# Retrieve and display the list of collections
collection = client.get_collection(collection_name)
collection

In [28]:
# Insert vectors into a collection
points=[
    models.PointStruct(
        id=1,
        vector=[0.9, 0.1, 0.1, 0.8], # High affordability, high innovation
        payload={"name": "Budget Smartphone", "category": "electronics", "price": 299},
    ),
    models.PointStruct(
        id=2,
        vector=[0.2, 0.9, 0.8, 0.5], # High quality, high popularity
        payload={"name": "Bestselling Novel", "category": "books", "price": 19},
    ),
    models.PointStruct(
        id=3,
        vector=[0.8, 0.3, 0.2, 0.9], # High affordability, high innovation (similar to ID 1)
        payload={"name": "Smart Home Hub", "category": "electronics", "price": 89},
    ),
    models.PointStruct(
        id=4,
        vector=[0.3, 0.5, 0.2, 0.7],
        payload={"name": "Iphone", "category": "electronics", "price": 1000},
    ),
    models.PointStruct(
        id=5,
        vector=[0.6, 0.7, 0.4, 0.9],
        payload={"name": "House", "category": "property", "price": 200000},
    ),
]

client.upsert(collection_name=collection_name, points=points)


UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
# Retrive specific collection details
collection_info = client.get_collection(collection_name)
collection_info

In [29]:
# Run the similarity search using vector
query_vector = [0.85, 0.2, 0.1, 0.9]
search_results = client.query_points(
    collection_name=collection_name,
    query=query_vector,)

search_results
# Output starts with BUdget Smartphone

# FIlter the search (only for electronics)
filtered_search = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    query_filter=models.Filter(
        must=[models.FieldCondition(key="category", match=models.MatchValue(value="electronics"))]
    )
)

print("Filtered search results:", filtered_search)

Filtered search results: points=[ScoredPoint(id=1, version=3, score=0.9933038, payload={'name': 'Budget Smartphone', 'category': 'electronics', 'price': 299}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3, version=3, score=0.9928858, payload={'name': 'Smart Home Hub', 'category': 'electronics', 'price': 89}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=4, version=3, score=0.85651433, payload={'name': 'Iphone', 'category': 'electronics', 'price': 1000}, vector=None, shard_key=None, order_value=None)]
